# Teleoperaattorin tulovarmistuksen yhteenvetokuution rakentaminen PROC SUMMARY -proseduurilla

## Yhteenveto

Langattoman operaattorin tulovarmistustiimi esikokoaa kuukauden tilaajapäiväkohtaiset laskutustietueet kompaktiksi yhteenvetokuutioksi, jotta analyytikot voivat porautua toteutuneeseen liikevaihtoon alueen, liittymätason ja palvelutyypin mukaan ilman raakafaktataulun uudelleenskannausta. `PROC SUMMARY` kokoaa 100 tilaajapäivätietuetta moniulotteiseksi solujoukoksi: hienoin taso (alue x liittymätaso x palvelutyyppi) täyttää 25 mahdollisesta 27 solusta, kun taas nimetyt alikuutiot antavat marginaalit, joita analyytikot kysyvät useimmin. Tässä esimerkkikuukaudessa operaattori laskutti 222,78 dollaria kolmella aktiivisella alueella, joista Etelä (97,44 dollaria) ja Itä (86,94 dollaria) muodostivat yhdessä 83 % liikevaihdosta ja Pohjoinen jäi viimeiseksi 38,40 dollarilla. Rikkain yksittäinen alikuutio on Plus-tason puhepalvelu (59,06 dollaria 18 tietueen yli), ja solujen järjestäminen tietuekohtaisen tuoton mukaan nostaa data-käyttösolut arvokkaimmiksi kohteiksi vuototarkastukselle (paras tuotto 7,87 dollaria/tietue). Jokainen alla oleva luku on luettu suoraan suoritetusta tulosteesta.

## Tietolähteet

| Aineisto | Rivit | Taso | Avainmuuttujat |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Yksi rivi per tilaajapäiväkohtainen käyttöyhteenveto | `region` (Itä/Etelä/Pohjoinen), `plan_tier` (Prepaid/Perus/Plus), `call_type` (Puhe/Tekstiviesti/Data), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Yksi rivi per ei-tyhjä (alue x liittymätaso x palvelutyyppi) -solu | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Yksi rivi per nimettyjen porauskuutioiden solu | `_TYPE_`, `_FREQ_`, `rev_total` |

Kaikki data luodaan sisäisesti `call streaminit()` / `rand()` -funktioilla; ulkoisia tiedostoja tai verkkoyhteyttä ei tarvita. Tämä ympäristö toimii lisenssittömässä tilassa, joten kirjoitetut aineistot on rajattu 100 havaintoon — skenaario on mitoitettu niin, että kuutio täyttyy kokonaan tämän rajan sisällä.

## Raakapuhelutietueista porattavaan kuutioon

Langattomat operaattorit laskuttavat liikevaihtoa miljoonista päivittäisistä käyttötapahtumista. Jotta tulovarmistuksen analyytikot voivat vastata kysymyksiin kuten *"Mikä oli Plus-tason puhepalvelun liikevaihto Etelä-alueella viime kuussa?"* ilman raakafaktataulun jatkuvaa uudelleenskannausta, **esikokoamme** datan kompaktiksi yhteenvetokuutioksi.

`PROC SUMMARY` on Base SAS:n perustyökalu juuri tähän: se ryhmittelee litteän faktataulun yhden tai useamman `CLASS`-ulottuvuuden mukaan ja kirjoittaa pyydetyt tunnusluvut tulosaineistoon, merkiten jokaisen rivin koodeilla `_TYPE_` (mitkä ulottuvuudet ovat aktiivisia) ja `_FREQ_` (solun takana olevat tietueet). Tämä tulosaineisto *on* moniulotteinen kuutio — sama koostumisrakenne, jonka OLAP-työkalu tarjoaisi, mutta toteutettuna tavallisena SAS-aineistona, jota voi tulostaa, yhdistää ja viipaloida.

Tämä muistikirja:

1. Luo realistisen kuukauden tilaajapäiväkohtaisia laskutustietueita.
2. Rakentaa hienoimman tason kuution (alue x liittymätaso x palvelutyyppi) `PROC SUMMARY NWAY` -proseduurilla.
3. Materialisoi nimetyt **porauksen alikuutiot** `TYPES`-lauseella.
4. Projisoi liikevaihdon tilaajakantaan `WEIGHT`-lauseella, poraa yhteen alueeseen ja järjestää solut tietuekohtaisen tuoton mukaan vuotojen tunnistamiseksi.

## Vaihe 1 - Synteettisen puhelutieto-/laskutusdatan luonti

Jokainen rivi tiivistää yhden tilaajan yhden palvelun käytön yhtenä päivänä. Käytämme `call streaminit` -funktiota toistettavuuden vuoksi ja `rand()`-funktiota uskottavien jakaumien arpomiseen: liikevaihto ja käyttö skaalautuvat liittymätason mukaan, puhepalvelun liikevaihto seuraa laskutettavia minuutteja ja datapalvelun liikevaihto seuraa megatavuja. Jokaiselle `RAND('table', ...)`-kutsulle annetaan yksi todennäköisyys per kategoria, jotta jokainen alue, taso ja palvelutyyppi esiintyy 100 tietueen otoksessa. Mukaan liitetään pieni `subscriber_wt`-kyselypaino, jotta voimme myöhemmin havainnollistaa painotettua mittaria.

In [1]:
TIEDOT work.cdr_billing;
    CALL streaminit(20260131);
    PITUUS region $12 plan_tier $9 call_type $14 device_class $16 bill_month $7;
    NIMIKE revenue       = "Toteutunut liikevaihto (USD)"
          call_minutes  = "Laskutettavat puheminuutit"
          data_mb       = "Datamäärä (Mt)"
          subscriber_wt = "Tilaajan kyselypaino";

    TEE i = 1 ASTI 100;
        /* --- Ulottuvuudet: yksi todennäköisyys per taso, summa 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        VALITSE (r);
            KUN (1) region = "Itä";
            KUN (2) region = "Etelä";
            MUULLOIN region = "Pohjoinen";
        LOPPU;

        p = rand("table", 0.30, 0.40, 0.30);
        VALITSE (p);
            KUN (1) plan_tier = "Prepaid";
            KUN (2) plan_tier = "Perus";
            MUULLOIN plan_tier = "Plus";
        LOPPU;

        c = rand("table", 0.50, 0.30, 0.20);
        VALITSE (c);
            KUN (1) call_type = "Puhe";
            KUN (2) call_type = "Tekstiviesti";
            MUULLOIN call_type = "Data";
        LOPPU;

        JOS rand("uniform") < 0.55 NIIN device_class = "Älypuhelin";
        MUUTEN device_class = "Peruspuhelin";

        bill_month = "2026-01";

        /* --- Mittarit, skaalattu tason ja palvelun mukaan --- */
        tier_mult = (plan_tier = "Prepaid")*0.7
                  + (plan_tier = "Perus")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Puhe")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Data")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "Tekstiviesti") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        TULOSTE;
    LOPPU;
    POISTA i r p c tier_mult base_rev;
SUORITA;

PROC PRINT TIEDOT=work.cdr_billing(obs=8) NIMIKE noobs;
    NIMIKE region='Alue' plan_tier='Liittymätyyppi' call_type='Palvelutyyppi'
          device_class='Laitetyyppi' bill_month='Laskutuskuukausi';
    OTSIKKO "Esimerkki tilaajapäiväkohtaisista laskutustiedoista";
SUORITA;


                                  Esimerkki tilaajapäiväkohtaisista laskutustiedoista                                   

     Alue   Liittymätyyppi  Palvelutyyppi   Laitetyyppi  Laskutuskuukausi  Laskutettavat puheminuutit     Datamäärä (Mt)  Toteutunut liikevaihto (USD)  Tilaajan kyselypaino
Pohjoinen  Plus             Tekstiviesti   Älypuhelin    2026-01                                    0                  0                          0.67                  1.13
Etelä      Prepaid          Tekstiviesti   Peruspuhelin  2026-01                                    0                  0                          0.94                  1.42
Etelä      Plus             Tekstiviesti   Älypuhelin    2026-01                                    0                  0                          0.99                  0.86
Etelä      Plus             Tekstiviesti   Älypuhelin    2026-01                                    0                  0                          1.01                  1.03
Etelä      Pl


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Vaihe 2 - Hienoimman tason kuution rakentaminen PROC SUMMARY NWAY -proseduurilla

`NWAY` säilyttää vain kaikkien `CLASS`-ulottuvuuksien yksityiskohtaisimman yhdistelmän - tässä jokaisen täytetyn (alue x plan_tier x call_type) -solun. Jokaiselle solulle tallennamme liikevaihdon `SUM`-, `MEAN`- ja `MAX`-tunnusluvut sekä minuuttien ja megatavujen summat, jotta analyytikko voi lukea solukohtaisen kokonaisliikevaihdon, johtaa tietuekohtaisen keskiarvon ja havaita suurimman yksittäisen veloituksen. `_FREQ_` kertoo, kuinka moni tilaajapäivä on kunkin solun takana. Pudotamme `_TYPE_`:n tässä, koska NWAY-tasolla jokaisella rivillä on sama tyyppi.

In [2]:
PROC SUMMARY TIEDOT=work.cdr_billing NWAY;
    LUOKKA region plan_tier call_type;
    MUUTTUJA revenue call_minutes data_mb;
    TULOSTE out=work.cube_nway(POISTA=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
SUORITA;

PROC PRINT TIEDOT=work.cube_nway(obs=12) NIMIKE noobs;
    NIMIKE region='Alue' plan_tier='Liittymätyyppi' call_type='Palvelutyyppi'
          _freq_='Frekvenssi' rev_total='Liikevaihto yhteensä (USD)'
          rev_mean='Keskimääräinen liikevaihto (USD)' rev_max='Suurin liikevaihto (USD)'
          min_total='Puheminuutit yhteensä' data_total='Data yhteensä (Mt)';
    OTSIKKO "NWAY-kuution solut (alue * liittymätyyppi * palvelutyyppi)";
    MUOTO rev_total rev_mean rev_max min_total data_total comma10.2;
SUORITA;


                               NWAY-kuution solut (alue * liittymätyyppi * palvelutyyppi)                               

  Alue   Liittymätyyppi  Palvelutyyppi  Frekvenssi   Liikevaihto yhteensä (USD)     Keskimääräinen liikevaihto (USD)  Suurin liikevaihto (USD)   Puheminuutit yhteensä   Data yhteensä (Mt)
Etelä   Perus            Data                    3                        12.02                                 4.01                      5.98                    0.00             1,368.50
Etelä   Perus            Puhe                    9                        22.51                                 2.50                      4.76                  451.80                 0.00
Etelä   Perus            Tekstiviesti            2                         2.01                                 1.00                      1.14                    0.00                 0.00
Etelä   Plus             Data                    2                        11.92                                 5.96          


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Vaihe 3 - Nimettyjen porauksen alikuutioiden materialisointi TYPES-lauseella

OLAP-kuutio esitallentaa koosteet, joita analyytikot selaavat eniten. `TYPES`-lause tekee juuri tämän: jokainen termi pyytää `PROC SUMMARY`:a tuottamaan yhden alikuution. Pyydämme kokonaissumman `()`, `region`-marginaalin sekä kaksisuuntaiset `region*plan_tier`- ja `call_type*plan_tier`-alikuutiot - polkuja, joita liikevaihtodashboard tarjoaisi. SAS merkitsee jokaisen alikuution `_TYPE_`-koodilla (bittimaski `CLASS`-listan yli), joten yksi tulosaineisto sisältää hierarkian jokaisen tason.

In [3]:
PROC SUMMARY TIEDOT=work.cdr_billing;
    LUOKKA region plan_tier call_type;
    MUUTTUJA revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    TULOSTE out=work.cube_hier
           sum(revenue)=rev_total;
SUORITA;

PROC PRINT TIEDOT=work.cube_hier NIMIKE noobs;
    NIMIKE region='Alue' plan_tier='Liittymätyyppi' call_type='Palvelutyyppi'
          _freq_='Frekvenssi' rev_total='Liikevaihto yhteensä (USD)';
    OTSIKKO "Alikuutioiden koosteet: kokonaissumma, alue, alue*taso, palvelutyyppi*taso";
    MUUTTUJA _type_ region plan_tier call_type _freq_ rev_total;
    MUOTO rev_total comma10.2;
SUORITA;


                       Alikuutioiden koosteet: kokonaissumma, alue, alue*taso, palvelutyyppi*taso                       

_TYPE_       Alue   Liittymätyyppi  Palvelutyyppi  Frekvenssi   Liikevaihto yhteensä (USD)
     0                                                    100                       222.78
     3             Perus            Data                    8                        38.06
     3             Perus            Puhe                   20                        42.33
     3             Perus            Tekstiviesti            8                         8.03
     3             Plus             Data                    3                        17.46
     3             Plus             Puhe                   18                        59.06
     3             Plus             Tekstiviesti           13                        11.75
     3             Prepaid          Data                    7                        14.50
     3             Prepaid          Puhe                   


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Vaihe 4 - Painotettu projektio, alueellinen poraus ja vuototarkastus

Kolme lukutapaa, joita tulovarmistustiimi todella tekee kuutiota vasten:

- **Painotettu projektio.** `WEIGHT=subscriber_wt` liitettynä `region*plan_tier`-yhteenvetoon raportoi liikevaihdon skaalattuna koko tilaajakantaan, jota otos edustaa, eikä pelkkiin raakoihin otosriveihin.
- **Poraus.** NWAY-kuution suodattaminen yhteen alueeseen laajentaa hierarkian yhden haaran - tässä Etelän - taso- ja palvelukohtaiseksi yksityiskohdaksi.
- **Vuototarkastus.** Solujen järjestäminen tietuekohtaisen keskiliikevaihdon mukaan nostaa esiin korkeimman tuoton solut; harvatietoiset, korkean tuoton solut ovat juuri niitä, joita tarkastus tutkii virheellisesti hinnoitellun tai vuotavan liikevaihdon varalta.

In [4]:
/* Painotettu liikevaihto projisoituna tilaajakantaan */
PROC SUMMARY TIEDOT=work.cdr_billing NWAY;
    LUOKKA region plan_tier;
    MUUTTUJA revenue;
    weight subscriber_wt;
    TULOSTE out=work.cube_wt(POISTA=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
SUORITA;

PROC PRINT TIEDOT=work.cube_wt NIMIKE noobs;
    NIMIKE region='Alue' plan_tier='Liittymätyyppi'
          rev_weighted='Painotettu liikevaihto (USD)' records='Tietueet';
    OTSIKKO "Painotettu liikevaihto alueittain * liittymätasoittain (projisoitu tilaajakantaan)";
    MUOTO rev_weighted comma10.2;
SUORITA;

/* Poraus Etelä-alueen haaraan kuutiossa */
PROC PRINT TIEDOT=work.cube_nway NIMIKE noobs;
    MISSÄ region = "Etelä";
    NIMIKE plan_tier='Liittymätyyppi' call_type='Palvelutyyppi' _freq_='Frekvenssi'
          rev_total='Liikevaihto yhteensä (USD)' rev_mean='Keskimääräinen liikevaihto (USD)';
    OTSIKKO "Poraus Etelään: liikevaihtosolut tasoittain ja palvelutyypeittäin";
    MUUTTUJA plan_tier call_type _freq_ rev_total rev_mean;
    MUOTO rev_total rev_mean comma10.2;
SUORITA;

/* Solujen järjestys tietuekohtaisen tuoton mukaan */
PROC SORT TIEDOT=work.cube_nway out=work.cube_ranked;
    MUKAAN LASKEVA rev_mean;
SUORITA;

PROC PRINT TIEDOT=work.cube_ranked(obs=6) NIMIKE noobs;
    NIMIKE region='Alue' plan_tier='Liittymätyyppi' call_type='Palvelutyyppi'
          _freq_='Frekvenssi' rev_mean='Keskimääräinen liikevaihto (USD)'
          rev_max='Suurin liikevaihto (USD)';
    OTSIKKO "Korkeimman keskiliikevaihdon solut (tietuekohtainen tuotto)";
    MUUTTUJA region plan_tier call_type _freq_ rev_mean rev_max;
    MUOTO rev_mean rev_max comma10.2;
SUORITA;


                   Painotettu liikevaihto alueittain * liittymätasoittain (projisoitu tilaajakantaan)                   

     Alue   Liittymätyyppi  Painotettu liikevaihto (USD)  Tietueet
Etelä      Perus                                   58.63        14
Etelä      Plus                                    56.29        15
Etelä      Prepaid                                 27.77        10
Itä        Perus                                   50.85        15
Itä        Plus                                    59.59        12
Itä        Prepaid                                 29.77        11
Pohjoinen  Perus                                   18.30         7
Pohjoinen  Plus                                    22.42         7
Pohjoinen  Prepaid                                 20.56         9

                           Poraus Etelään: liikevaihtosolut tasoittain ja palvelutyypeittäin                            

 Liittymätyyppi  Palvelutyyppi  Frekvenssi   Liikevaihto yhteensä (USD)     Keskimäär


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Tulosten tulkinta

Yhteenvetokuutio muuttaa 100 raakaa tilaajapäivärivi kompaktiksi joukoksi esikoottuja soluja, jotka vastaavat porauskysymyksiin heti, ilman faktataulun uudelleenskannausta:

- **Missä liikevaihto sijaitsee.** `region`-marginaali (`_TYPE_=4`) näyttää, että Etelä laskutti 97,44 dollaria ja Itä 86,94 dollaria - yhdessä 83 % kuukauden 222,78 dollarista - kun taas Pohjoinen toi 38,40 dollaria. `call_type*plan_tier`-alikuutiossa (`_TYPE_=3`) Plus-tason puhepalvelu on yksittäin rikkain solu, 59,06 dollaria 18 tietueen yli, ja seuraavana Perus-tason puhepalvelu 42,33 dollarilla.
- **Painotettu projektio.** Kyselypainon soveltaminen järjestää ranking-listan uudelleen kohti liittymiä, joiden tilaajilla on enemmän painoarvoa: Itä-Plus (59,59 dollaria) ja Etelä-Perus (58,63 dollaria) johtavat projisoitua `region*plan_tier`-liikevaihtoa, eri kuva kuin painottamattomat aluesummat, ja muistutus siitä, että altistusta mitoittaessa kannattaa raportoida projisoitu eikä otettu liikevaihto.
- **Tietuekohtainen tuotto ja vuototarkastus.** NWAY-solujen järjestäminen `rev_mean`:n mukaan nostaa data-käyttösolut kärkeen - Pohjoinen-Perus-Data (7,87 dollaria/tietue) ja Etelä-Plus-Data (5,96 dollaria/tietue) - mikä vahvistaa, että raskas datankäyttö tuottaa korkeimman tietuekohtaisen liikevaihdon. Harvatietoiset kärkisolut (yksi tai kaksi tietuetta) ovat juuri niitä soluja, joiden taustalla olevat puhelutietueet tulovarmistuksen analyytikko poimisi varmistaakseen, että korkea veloitus on oikein hinnoiteltu eikä virhe.

Tulovarmistustiimille tämä kuutio on varianssidashboardien perusta: verrataan toteutunutta liikevaihtoa odotettuun hinnastoliikevaihtoon per (alue, taso, palvelutyyppi) -solu, ja solut, joiden keskiarvo tai painotettu summa poikkeavat eniten, muodostavat auditoinnin arvoiset vuototapaukset. Koska koko rakenne on tavallinen SAS-aineisto, seuraavan kuukauden kuutio voidaan yhdistää (union), erotella tai liittää hinnastoon samoilla Base SAS -työkaluilla.